# Mining cross-page document-element sequences

This notebook demonstrates `archival_structures.analysis.sequence_patterns` (task 4): putting
every line of an inventory number into one global, cross-scan reading sequence
(`build_line_sequence`), counting recurring cluster-label n-grams (`find_ngram_patterns`), and
segmenting the sequence into document `Element`s -- including elements that span a page break
(`segment_into_elements`).

It compares two structurally different inventory numbers:

- `NL-HaNA_2.10.50_1` -- a tabular register (rows of cells, used as the "table" example
  throughout the other task 2/3 demo notebooks).
- `NL-AsnDA_0114.11_1` -- notary deeds (running prose, legal boilerplate).

The expectation: a table's rows shouldn't usually need to continue across a page break, while
a deed's prose plausibly does (e.g. a clause that runs out of room at the bottom of one page and
finishes at the top of the next). Both inventories are split into single pages first (task 1)
and line-clustered (task 3) before sequence mining, exactly as the other task 2/3 demo
notebooks do.

In [1]:
from pathlib import Path

import pandas as pd
import pagexml.parser as pagexml_parser

from archival_structures.analysis.opening_detection import classify_inventory_structure, split_inventory_into_pages
from archival_structures.analysis.line_clustering import extract_corpus_line_features, cluster_lines
from archival_structures.analysis.sequence_patterns import build_line_sequence, find_ngram_patterns, segment_into_elements

SAMPLE_SIZE = 80  # number of scans to process per inventory (kept small for a quick demo run)
PAGEXML_ROOT = Path('../../data/PageXML')

## Load, classify, and split both inventories

`NL-AsnDA_0114.11_1` needs a wrinkle: only its first and last scan (out of 630) are single-page
covers, the other 628 are openings -- a far more lopsided single/opening ratio than the
inventories `classify_inventory_structure`'s default `min_share=0.15` was validated against (it
requires *each* width group to be at least 15% of all scans). With the default, the 2 covers
(0.3%) don't clear that bar and the whole inventory comes back `mixed`. Lowering `min_share` (and
classifying on the *full* 630-scan corpus, not just the `SAMPLE_SIZE` subset processed below --
otherwise there's no cover scan in the sample to detect at all) fixes this; see
[`docs/findings.md`](../../docs/findings.md) for the full explanation. `NL-HaNA_2.10.50_1` needs
no such adjustment -- its covers are a large enough share of the (much shorter) inventory for the
defaults to work directly, as in the other demo notebooks.

In [2]:
def load_pages(institute, archive, inventory_num, sample_size=SAMPLE_SIZE, min_share=0.15):
    pagexml_dir = PAGEXML_ROOT / institute / archive / inventory_num
    all_xml_paths = sorted(pagexml_dir.glob('*.xml'))
    all_scans = [pagexml_parser.parse_pagexml_file(str(p)) for p in all_xml_paths]
    # classify on the full corpus so rare cover scans are still detected as a real minority
    # group, even though only a sample of scans gets split/processed below
    structure = classify_inventory_structure(all_scans, min_share=min_share)
    print(f"{inventory_num}: {len(all_scans)} scans, structure={structure.structure_type} {structure.type_counts}")

    sample_scans = all_scans[:sample_size]
    all_pages = split_inventory_into_pages(sample_scans, structure=structure)
    pages = [page for page in all_pages if len(page.get_lines()) > 0]
    print(f"  -> sampled {len(sample_scans)} scans -> {len(all_pages)} pages -> {len(pages)} with at least one line")
    return pages


table_pages = load_pages('NL-HaNA', 'NL-HaNA_2.10.50', 'NL-HaNA_2.10.50_1')
deed_pages = load_pages('NL-AsnDA', 'NL-AsnDA_0114.11', 'NL-AsnDA_0114.11_1', min_share=0.002)

NL-HaNA_2.10.50_1: 336 scans, structure=book_of_openings {'opening': 242, 'single_page': 88, 'odd_shaped': 6}


  -> sampled 80 scans -> 138 pages -> 133 with at least one line


NL-AsnDA_0114.11_1: 630 scans, structure=book_of_openings {'opening': 628, 'single_page': 2}


  -> sampled 80 scans -> 159 pages -> 125 with at least one line


## Cluster lines and build the line sequence

`build_line_sequence` orders every line of every page into one global document-order sequence
(trusting the PageXML's own reading order within each page), carrying along each line's cluster
label from `line_clustering.cluster_lines`.

In [3]:
def cluster_and_sequence(pages):
    line_df = extract_corpus_line_features(pages)
    line_df['cluster'] = cluster_lines(line_df, features=('left_norm', 'width_norm'), min_cluster_size=15)
    print(f"  {len(line_df)} lines, {line_df['cluster'].nunique()} clusters "
         f"({(line_df['cluster'] == -1).sum()} noise)")
    line_clusters = line_df.set_index(['scan_id', 'line_id'])['cluster']
    return build_line_sequence(pages, line_clusters)


print('table register:')
table_sequence = cluster_and_sequence(table_pages)
print('notary deeds:')
deed_sequence = cluster_and_sequence(deed_pages)

table register:
  8333 lines, 33 clusters (2136 noise)


notary deeds:


  7736 lines, 28 clusters (1265 noise)


## Recurring n-gram patterns

`find_ngram_patterns` counts consecutive cluster-label bigrams/trigrams across the whole
sequence (not broken at page boundaries). The most frequent patterns should be runs of the same
line type repeating -- e.g. several body-text lines of the same column/style in a row.

In [4]:
def show_top_ngrams(sequence, label, n=10):
    ngrams = find_ngram_patterns(sequence, n=(2, 3))
    print(f"{label}: top {n} of {len(ngrams)} distinct n-grams")
    for ngram, count in ngrams.most_common(n):
        print(f"  {ngram}  {count}")
    return ngrams


table_ngrams = show_top_ngrams(table_sequence, 'table register')
print()
deed_ngrams = show_top_ngrams(deed_sequence, 'notary deeds')

table register: top 10 of 641 distinct n-grams
  (np.int64(5), np.int64(5))  556
  (np.int64(28), np.int64(28))  504
  (np.int64(3), np.int64(3))  482
  (np.int64(5), np.int64(5), np.int64(5))  418
  (np.int64(3), np.int64(3), np.int64(3))  406
  (np.int64(28), np.int64(28), np.int64(28))  342
  (np.int64(10), np.int64(10))  268
  (np.int64(7), np.int64(7))  253
  (np.int64(27), np.int64(27))  237
  (np.int64(6), np.int64(6))  219

notary deeds: top 10 of 517 distinct n-grams
  (np.int64(14), np.int64(14))  2521
  (np.int64(14), np.int64(14), np.int64(14))  2076
  (np.int64(19), np.int64(19))  833
  (np.int64(19), np.int64(19), np.int64(19))  762
  (np.int64(3), np.int64(3))  349
  (np.int64(3), np.int64(3), np.int64(3))  298
  (np.int64(26), np.int64(26))  146
  (np.int64(26), np.int64(26), np.int64(26))  120
  (np.int64(7), np.int64(7))  67
  (np.int64(3), np.int64(7))  67


## Segmenting into document elements -- and finding cross-page continuations

`segment_into_elements` run-length-segments same-label runs into `Element`s, using
`detect_cross_page_continuation` to decide whether a run that happens to straddle a page
boundary is genuinely one continuing element.

`detect_cross_page_continuation`'s `max_vertical_gap`/`max_horizontal_diff` are in **absolute
pixels** at the scan's own resolution. The default (`max_vertical_gap=150`) turns out to be far
too strict for these ~4000-7000px-wide scans -- a page's own top/bottom margins alone are
several hundred pixels, so a genuinely continuing line on the next page is already further than
150px away before any real gap is considered. Scaling the thresholds up to match the actual
image resolution (`max_vertical_gap=800, max_horizontal_diff=300`) is what actually surfaces
real cross-page elements below; see [`docs/findings.md`](../../docs/findings.md).

In [5]:
MAX_VERTICAL_GAP = 800
MAX_HORIZONTAL_DIFF = 300


def segment_and_summarise(sequence, pages, label):
    elements = segment_into_elements(sequence, pages, max_vertical_gap=MAX_VERTICAL_GAP,
                                     max_horizontal_diff=MAX_HORIZONTAL_DIFF)
    multi_span = [el for el in elements if len(el.spans) > 1]
    print(f"{label}: {len(elements)} elements, {len(multi_span)} span more than one page")
    return elements, multi_span


table_elements, table_multi_span = segment_and_summarise(table_sequence, table_pages, 'table register')
deed_elements, deed_multi_span = segment_and_summarise(deed_sequence, deed_pages, 'notary deeds')

table register: 2666 elements, 0 span more than one page


notary deeds: 2359 elements, 34 span more than one page


As expected: the table register's rows stay within one page (0 cross-page elements, even with
the same generous thresholds used for the deeds below -- this isn't a threshold artefact, rows
genuinely don't span a break), while the notary deeds show real cross-page continuations. Look
at the actual text of one to confirm it's a genuine continuation, not a coincidence.

In [6]:
lines_by_id = {(page.id, line.id): line for page in deed_pages for line in page.get_lines()}

example = deed_multi_span[0]
print(f"element type (cluster label): {example.element_type}\n")
for span in example.spans:
    print(f"--- {span.scan_id} ---")
    for line_id in span.line_ids:
        print(' ', lines_by_id[(span.scan_id, line_id)].text)
    print()

element type (cluster label): 14

--- NL-AsnDA_0114.11_1_0005.jpg-verso ---
  Dat de schuldenaar de verbonden goederen niet
  in waarde zal mogen verminderen en net voor
  langer dan eon jaar zal mogen verhween noch huur
  penningen bij vooruitbeleing zal mogen ontvangen
  zonder schriftelijke toestemming van den schuldescher
  Dat bij geledle of gedeeltelijke vervreemding van
  het verbonden, wanmeer het van bedemming mocht
  worden veranderd) met rettere hjpotheek of wel met
  eenig ander zakelijk recht bezwaard, zoo het verbon
  dine in waarde mocht verminderen of in beslag
  mocht worden genomen, hoofdsom eerden en hes
  ten dadelijk vorderbaar en operschlaar zijn, ge„
  lijk mede indien de schuldenaar wordt verklaard te
  zijn in staat van zuillissement of hennelijk onver-
  mogen, bij vrijwillige of gerechtelijke boedelafstand
  of nalatig is in de behoorlijke voldoening van kepi„
  tuul of rente en eindelijk bij niet nakoming of over
  bieding van eind meer der overige door den 

## Summary

| | table register (`NL-HaNA_2.10.50_1`) | notary deeds (`NL-AsnDA_0114.11_1`) |
|---|---|---|
| pages processed | see counts above | see counts above |
| elements found | see counts above | see counts above |
| elements spanning >1 page | 0 | several -- genuine continuing clauses |

The table register's rows never need to continue across a page break; the notary deeds' prose
does, but only becomes visible once the cross-page distance thresholds are scaled to the scans'
actual pixel resolution rather than left at the (too-strict-by-default) `max_vertical_gap=150`.
